# Step 4 · Evals: proving it works
Compare the tool's verdicts with an **answer key** (`evals/golden_set.json`) and grade chat answers with **Ragas**,
using a judge model from a *different* family.

> **Start Jupyter through the project script** so it gets the same settings as the app:
>
> ```bash
> cd copilot
> ./scripts/run.sh jupyter lab
> ```

In [ ]:
import sys, os
sys.path.insert(0, "..")          # make the copilot's `app` package importable
os.chdir("..")                    # run from copilot/, like the app does
print("Models are called at:", os.getenv("BIFROST_URL"), "| profile:", os.getenv("SETTINGS_PROFILE") or "direct to Ollama")

## 1. The answer key

In [ ]:
import json
golden = json.loads(open("evals/golden_set.json").read())
[(c["id"], c["expected_score"], c["expected_band"]) for c in golden["contracts"]]

## 2. Evaluate one contract (review + chat + Ragas: about 10–15 minutes)

In [ ]:
from evals.run_evals import evaluate
lexmore = next(c for c in golden["contracts"] if c["id"] == "lexmore")
run = evaluate(lexmore, golden["questions"], skip_ragas=False)
run["score"], run["band"], f"{run['exact']}/10 exact", run["seconds"]

In [ ]:
for r in run["rules"]:
    print(f"{r['rule']:<4} expected {r['expected']:<12} got {r['got']:<12} {'✓' if r['expected'] == r['got'] else '✗'}")

## 3. Targets and the release blocker
A risky clause marked OK (`compliant_on_risky`) must always be 0.

In [ ]:
from evals import metrics
result = metrics.summarise([run])
for t in result["targets"]:
    print(f"{t['metric']:<26} target {t['target']:<6} actual {t['actual']}")
print("compliant_on_risky =", result["compliant_on_risky"])

The Ragas scores are also attached to each chat trace in **Langfuse**. For all five contracts: `make evals`.